In [1]:
import numpy as np
import time

print("=== DEPLOYING PRODUCTION RAG PIPELINE ON AWS t3.2xlarge ===")

# Simulated Source Document Context (Infrastructure & Cost Data)
raw_document = """
[INFRA-DATA-2026] Our current operational AI infrastructure cluster runs on a shared AWS t3.2xlarge instance costing $11.65. 
Persistent storage is allocated at 200 GB costing $16.00. The API Gateway operates via a dedicated proxy for $7.00. 
Model token pools consume $315.00, bringing our total baseline program operational cost to exactly $349.65 per month.
"""

=== DEPLOYING PRODUCTION RAG PIPELINE ON AWS t3.2xlarge ===


In [2]:
# STEP 1: RECURSIVE CHARACTER CHUNKING SIMULATION
print("\n--- Step 1: Executing Structural Text Chunking ---")
def simple_recursive_chunker(text, max_chars=120):
    # Splits by sentences first to preserve semantic bounds
    sentences = text.strip().split(". ")
    chunks = []
    current_chunk = ""
    
    for sentence in sentences:
        if len(current_chunk) + len(sentence) <= max_chars:
            current_chunk += sentence + ". "
        else:
            if current_chunk: chunks.append(current_chunk.strip())
            current_chunk = sentence + ". "
    if current_chunk:
        chunks.append(current_chunk.strip())
    return chunks

document_chunks = simple_recursive_chunker(raw_document)
for idx, chunk in enumerate(document_chunks):
    print(f" Chunk [{idx}]: \"{chunk}\" (Length: {len(chunk)} chars)")


--- Step 1: Executing Structural Text Chunking ---
 Chunk [0]: "[INFRA-DATA-2026] Our current operational AI infrastructure cluster runs on a shared AWS t3.2xlarge instance costing $11.65." (Length: 124 chars)
 Chunk [1]: "Persistent storage is allocated at 200 GB costing $16.00. The API Gateway operates via a dedicated proxy for $7.00." (Length: 115 chars)
 Chunk [2]: "Model token pools consume $315.00, bringing our total baseline program operational cost to exactly $349.65 per month.." (Length: 118 chars)


In [3]:
# STEP 2: VECTORIZATION & EMBEDDING GENERATION (768-Dimensional)
print("\n--- Step 2: Vectorizing Chunks into High-Dimensional Space ---")
# To simulate real-world embedding sizes (like huggingface bge-base or BERT models)
DIMENSIONS = 768

def generate_mock_embedding(text_content):
    # Deterministic vector seed based on character values to simulate an embedding model run on CPU
    seed = sum(ord(char) for char in text_content) % 1000
    np.random.seed(seed)
    raw_vector = np.random.randn(DIMENSIONS)
    # L2 Normalization so dot product equals Cosine Similarity perfectly
    normalized_vector = raw_vector / np.linalg.norm(raw_vector)
    return normalized_vector

# Vector database storage representation
vector_database = {}
for idx, chunk in enumerate(document_chunks):
    vector_database[idx] = {
        "text": chunk,
        "embedding": generate_mock_embedding(chunk)
    }
print(f"Successfully processed and stored {len(vector_database)} chunks into the vector store registry.")


--- Step 2: Vectorizing Chunks into High-Dimensional Space ---
Successfully processed and stored 3 chunks into the vector store registry.


In [4]:
# STEP 3: THE RETRIEVAL PHASE (Vector Database Query)
print("\n--- Step 3: Querying the Vector Database via Cosine Similarity ---")
user_query = "What is the total combined cost of our program infrastructure?"
print(f"User Question: '{user_query}'")

query_vector = generate_mock_embedding(user_query)

best_score = -1
retrieved_context = ""

for idx, record in vector_database.items():
    # Because our vectors are normalized, dot product calculated here is the Cosine Similarity
    similarity_score = np.dot(query_vector, record["embedding"])
    print(f" -> Comparing against Chunk [{idx}] | Similarity Score: {similarity_score:.4f}")
    
    if similarity_score > best_score:
        best_score = similarity_score
        retrieved_context = record["text"]

print(f"\n[WINNING CHUNK RETRIEVED]: \"{retrieved_context}\"")


--- Step 3: Querying the Vector Database via Cosine Similarity ---
User Question: 'What is the total combined cost of our program infrastructure?'
 -> Comparing against Chunk [0] | Similarity Score: 0.0202
 -> Comparing against Chunk [1] | Similarity Score: 0.0231
 -> Comparing against Chunk [2] | Similarity Score: 0.0598

[WINNING CHUNK RETRIEVED]: "Model token pools consume $315.00, bringing our total baseline program operational cost to exactly $349.65 per month.."


In [5]:
# STEP 4: GENERATION PREPARATION (Grounding Context Formulation)
print("\n--- Step 4: Framing Grounded Prompt for the API Gateway ---")
grounded_system_prompt = f"""
You are a financial infrastructure auditor running on a t3.2xlarge cluster.
Answer the user's query using ONLY the verified source context below. If the answer cannot be calculated from the context, say 'Missing Data'.

[VERIFIED SOURCE CONTEXT]:
{retrieved_context}

[USER QUESTION]:
{user_query}
"""
print("Final Context payload ready to route over API Gateway ($7.00/mo proxy channel):")
print(grounded_system_prompt)


--- Step 4: Framing Grounded Prompt for the API Gateway ---
Final Context payload ready to route over API Gateway ($7.00/mo proxy channel):

You are a financial infrastructure auditor running on a t3.2xlarge cluster.
Answer the user's query using ONLY the verified source context below. If the answer cannot be calculated from the context, say 'Missing Data'.

[VERIFIED SOURCE CONTEXT]:
Model token pools consume $315.00, bringing our total baseline program operational cost to exactly $349.65 per month..

[USER QUESTION]:
What is the total combined cost of our program infrastructure?

